# 10 - Dual-Layer Watermarking Example (uniform lists)

Runs **both** watermark layers on the same generation, plus each layer alone:

| | layer 1 (public topic) | layer 2 (private KGW) | dual |
|---|---|---|---|
| greenlist | static per-topic CSV (`data/greenlist/`) | derived from secret key + prev tokens | both boosts summed |
| selection | topic inferred from prompt | hash-based per position | - |
| logit update | `+= w1 * delta_public` | `+= w2 * delta_private` | `+= w1*delta_public + w2*delta_private` |

For every prompt the notebook emits **four** texts sharing the same routed topic:
`plain_output`, `layer1_only_output` (topic boost only), `layer2_only_output`
(KGW boost only), `dual_watermarked_output`.

Everything sits in the tokenizer id-space (`len(tokenizer)` = 50265): the uniform
lists were built there, `derive_set` permutes it, and both detectors assume it.

Flow:
1. config (`secondLayer.json`) + model + secret key
2. real uniform greenlists via `first_layer.load_uniform_greenlists` + topic geometry
3. `DualLayerWatermarkProcessor` — both boosts in one logits processor, `w1`/`w2` switches
4. `DualWaterMarking.watermark(prompts)` -> DataFrame with all four outputs
5. cross-detection: `src/detection/topic_detection` + `src/detection/kgw_detection`
   score every output -> dual text must trip BOTH alarms

In [1]:
!git clone https://github.com/pravaspaudel/Dual_watermarking_Scheme.git

Cloning into 'Dual_watermarking_Scheme'...
remote: Enumerating objects: 230, done.
remote: Counting objects: 100% (79/79), done.
remote: Compressing objects: 100% (63/63), done.
remote: Total 230 (delta 39), reused 50 (delta 16), pack-reused 151 (from 1)
Receiving objects: 100% (230/230), 43.50 MiB | 35.22 MiB/s, done.
Resolving deltas: 100% (98/98), done.


In [2]:
%cd Dual_watermarking_Scheme/

/content/Dual_watermarking_Scheme


In [3]:
import torch
from huggingface_hub import login
login()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


cuda


In [4]:
import pandas as pd

GREENLIST_DIR  = "data/greenlist"   # real uniform lists from 05_greenlist_construction.ipynb
SPLIT          = "all"              # topic-layer split boosted during generation
DELTA_TOPIC    = 2.0                # first-layer boost (pipeline operating point)
DELTA_PRIVATE  = 0.7                # second-layer boost (06/07 precedent)
W1, W2         = 1.0, 1.0           # dual weights: 1/1 = both layers full strength
MAX_NEW_TOKENS = 200                # length = detection power (see notebook 08)
SEED           = 0
torch.manual_seed(SEED)

## Step 1 - Reusable pieces from `src/`

- `first_layer.load_uniform_greenlists` / `build_topic_matrix` — same source of
  truth as generation and detection (notebook 08 / topic pipeline)
- `key_manager.derive_set` — the private KGW greenlist rule, shared with
  `second_layer.PrivateWatermarkProcessor` and `kgw_detection`
- `load_config` / `load_model` / `generate_key` — second-layer plumbing

No synthetic greenlists here: the per-topic CSVs under `data/greenlist/` are the
same public lists notebook 08 detects against.

In [5]:
import sys
sys.path.insert(0, "..")   # running from notebooks/
sys.path.insert(0, ".")    # running from repo root after %cd

from transformers import LogitsProcessor

from src.utils.loadConfig import load_config
from src.utils.key_manager import derive_set, generate_key
from src.watermark.first_layer import (
    TopicBoostProcessor,
    build_topic_matrix,
    load_uniform_greenlists,
)

In [6]:
config = load_config("secondLayer")
config

{'MODEL_NAME': 'facebook/opt-2.7b',
 'GREEN_FRACTION': 0.5,
 'PREV_TOKEN_SIZE': 5,
 'DETECTION_THRESHOLD': 0.6,
 'P_VALUE_THRESHOLD': 0.05}

In [7]:
from src.utils.model import load_model

model, tokenizer, VOCAB_SIZE = load_model(config["MODEL_NAME"])
model.to(device).eval()

# OPT quirk (see notebook 08): the embedding table has 7 extra reserved rows;
# lists / derive_set / detectors all live in the tokenizer id-space.
assert model.get_input_embeddings().weight.shape[0] >= VOCAB_SIZE

key = generate_key()          # owner-side secret for the KGW layer
print(f"secret key generated: {key[:16]}...")

config.json:   0%|          | 0.00/691 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 5.30GB            

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

model and tokenizer of facebook/opt-2.7b loaded with vocab_size 50265
secret key generated: 98fcdc8ab8802cc3...


In [8]:
greenlists = load_uniform_greenlists(GREENLIST_DIR)   # {topic: {"all": [...], "content": [...]}}
TOPICS = sorted(greenlists)
print({t: len(v[SPLIT]) for t, v in greenlists.items()})

emb = model.get_input_embeddings().weight.detach().float()
normed_embeddings = emb / emb.norm(dim=1, keepdim=True)
topic_matrix = build_topic_matrix(model, tokenizer, TOPICS)   # (K, d), unit rows

{'entertainment': 4244, 'finance': 4236, 'history': 5803, 'medicine': 4230, 'politics': 7701, 'science': 6092, 'sports': 6710, 'technology': 11240}


## Step 2 - The dual logits processor

One `LogitsProcessor` carries both layers; the `w1`/`w2` weights switch layers on
and off, so a single class produces layer-1-only (`w2=0`), layer-2-only (`w1=0`)
and dual text (`w1=w2=1`).

In [9]:
class DualLayerWatermarkProcessor(LogitsProcessor):
    """
    Applies both watermark boosts to the next-token logits at every step.
    boosted_logit = base_logit + w1 * delta_public + w2 * delta_private
    """

    def __init__(self, topic_green_ids, key, vocab_size, green_fraction=0.5,
                 delta_public=2.0, delta_private=0.7, w1=1.0, w2=1.0,
                 prev_token_size=5):
        self.topic_ids = torch.tensor(topic_green_ids, dtype=torch.long)
        self.key = key
        self.vocab_size = vocab_size
        self.green_fraction = green_fraction
        self.delta_public = delta_public
        self.delta_private = delta_private
        self.w1 = w1
        self.w2 = w2
        self.prev_token_size = prev_token_size

    def __call__(self, input_ids, scores):
        if self.w1 != 0.0:                       # layer 1: fixed topic greenlist
            scores[..., self.topic_ids.to(scores.device)] += self.w1 * self.delta_public

        if self.w2 != 0.0:                       # layer 2: keyed rolling greenlist
            for b in range(input_ids.shape[0]):
                preferred = derive_set(
                    vocab_size=self.vocab_size,
                    green_fraction=self.green_fraction,
                    key=self.key,
                    prev_tokens_size=self.prev_token_size,
                    prev_tokens=input_ids[b].tolist(),
                )
                idx = torch.tensor(list(preferred), dtype=torch.long,
                                   device=scores.device)
                scores[b, idx] += self.w2 * self.delta_private
        return scores

In [10]:
@torch.no_grad()
def rank_topics(prompt):
    """Cosine of mean-pooled prompt embedding vs each topic vector.

    Same routing math as first_layer.TopicWiseWatermarking.extract_topic.
    """
    ids = tokenizer.encode(prompt, add_special_tokens=False)
    vec = normed_embeddings[ids].mean(dim=0)
    vec = vec / vec.norm()
    scores = (topic_matrix @ vec).tolist()
    return sorted(zip(TOPICS, scores), key=lambda x: -x[1])


def extract_topic(prompt):
    ranked = rank_topics(prompt)
    return ranked[0][0], ranked

## Step 3 - End-to-end wrapper

`DualWaterMarking.watermark(prompts)` returns one DataFrame row per prompt with
plain / layer-1-only / layer-2-only / dual outputs. The already-built `greenlists`
and geometry are passed in to avoid reloading; if omitted the class rebuilds them
itself (handy once exported to `src/watermark/dual_layer.py`).

In [11]:
class DualWaterMarking:
    """End-to-end dual watermarking over the uniform static greenlists."""

    def __init__(self, model, tokenizer, key, greenlists=None,
                 greenlist_dir="data/greenlist", split="all",
                 normed_embeddings=None, topic_matrix=None,
                 green_fraction=0.5, prev_token_size=5,
                 delta_public=2.0, delta_private=0.7, w1=1.0, w2=1.0,
                 max_new_tokens=200, temperature=1.0, top_p=0.9):
        self.model = model.eval()
        self.tokenizer = tokenizer
        self.key = key
        self.split = split
        self.vocab_size = len(tokenizer)      # id-space shared by lists + detector
        self.green_fraction = green_fraction
        self.prev_token_size = prev_token_size
        self.delta_public = delta_public
        self.delta_private = delta_private
        self.w1, self.w2 = w1, w2
        self.max_new_tokens = max_new_tokens
        self.temperature = temperature
        self.top_p = top_p

        if greenlists is None:
            greenlists = load_uniform_greenlists(greenlist_dir)
        self.greenlists = greenlists
        self.topics = sorted(greenlists)

        if normed_embeddings is None:
            emb = self.model.get_input_embeddings().weight.detach().float()
            normed_embeddings = emb / emb.norm(dim=1, keepdim=True)
        self.normed_embeddings = normed_embeddings
        self.topic_matrix = (topic_matrix if topic_matrix is not None
                             else build_topic_matrix(self.model, self.tokenizer, self.topics))

    @torch.no_grad()
    def extract_topic(self, prompt):
        ids = self.tokenizer.encode(prompt, add_special_tokens=False)
        vec = self.normed_embeddings[ids].mean(dim=0)
        vec = vec / vec.norm()
        scores = (self.topic_matrix @ vec).tolist()
        ranked = sorted(zip(self.topics, scores), key=lambda x: -x[1])
        return ranked[0][0], ranked

    def _make_processor(self, topic, w1, w2):
        """Both layers in one processor; set a weight to 0 to drop that layer."""
        return DualLayerWatermarkProcessor(
            topic_green_ids=self.greenlists[topic][self.split],
            key=self.key,
            vocab_size=self.vocab_size,
            green_fraction=self.green_fraction,
            delta_public=self.delta_public,
            delta_private=self.delta_private,
            w1=w1, w2=w2,
            prev_token_size=self.prev_token_size,
        )

    def _generate(self, inputs, processor=None):
        kwargs = dict(max_new_tokens=self.max_new_tokens, do_sample=True,
                      temperature=self.temperature, top_p=self.top_p)
        if processor is not None:
            kwargs["logits_processor"] = [processor]
        with torch.no_grad():
            out = self.model.generate(**inputs, **kwargs)
        return self.tokenizer.decode(out[0], skip_special_tokens=True)

    def watermark(self, prompts, include_single_layers=True):
        """Plain vs layer1-only vs layer2-only vs dual, one row per prompt."""
        rows = []
        for i, prompt in enumerate(prompts):
            topic, ranked = self.extract_topic(prompt)
            inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)

            row = {
                "id": i,
                "prompt": prompt,
                "topic": topic,
                "topic_score": round(ranked[0][1], 4),
                "plain_output": self._generate(inputs),
                "dual_watermarked_output": self._generate(
                    inputs, self._make_processor(topic, self.w1, self.w2)),
            }
            if include_single_layers:
                row["layer1_only_output"] = self._generate(
                    inputs, self._make_processor(topic, 1.0, 0.0))
                row["layer2_only_output"] = self._generate(
                    inputs, self._make_processor(topic, 0.0, 1.0))
            rows.append(row)
        return pd.DataFrame(rows)


dwm = DualWaterMarking(
    model=model,
    tokenizer=tokenizer,
    key=key,
    greenlists=greenlists,
    split=SPLIT,
    normed_embeddings=normed_embeddings,
    topic_matrix=topic_matrix,
    green_fraction=config["GREEN_FRACTION"],
    prev_token_size=config["PREV_TOKEN_SIZE"],
    delta_public=DELTA_TOPIC,
    delta_private=DELTA_PRIVATE,
    w1=W1, w2=W2,
    max_new_tokens=MAX_NEW_TOKENS,
)
print(f"DualWaterMarking ready: {len(dwm.topics)} topics | "
      f"delta_public={DELTA_TOPIC} delta_private={DELTA_PRIVATE} "
      f"green_fraction={config['GREEN_FRACTION']}")

DualWaterMarking ready: 8 topics | delta_public=2.0 delta_private=0.7 green_fraction=0.5


In [12]:
sample_prompts = [
    "The recent breakthrough in quantum computing suggests that",
]

results_df = dwm.watermark(sample_prompts, include_single_layers=True)

pd.set_option("display.max_colwidth", None)
for _, r in results_df.iterrows():
    print("=" * 80)
    print(f"PROMPT : {r['prompt']}")
    print(f"TOPIC  : {r['topic']}  (score={r['topic_score']})")
    print(f"PLAIN  : {r['plain_output']}")
    print(f"LAYER1 : {r['layer1_only_output']}")
    print(f"LAYER2 : {r['layer2_only_output']}")
    print(f"DUAL   : {r['dual_watermarked_output']}")
print("=" * 80)

results_df

PROMPT : The recent breakthrough in quantum computing suggests that
TOPIC  : technology  (score=0.2509)
PLAIN  : The recent breakthrough in quantum computing suggests that the first super-powerful computers will exist in the next two decades, leading to a race between the US and China.

But, while it’s widely known that the US leads the world when it comes to military spending, few know that we are also leading in the science of science itself.

Quantum computing can solve certain tasks faster than current computers, and the potential of this technology is staggering. So how are the US and China going to develop it? And will quantum computers mean the end of the digital age?

Read more: China could overtake US to become world’s superpower by 2049

The technology

A quantum computer is essentially a computer with an unusually strong connection to the laws of nature.

A classical computer is made up of bits, which are usually represented by 0 or 1. Quantum bits are 0 and 1 at the same ti

,id,prompt,topic,topic_score,plain_output,dual_watermarked_output,layer1_only_output,layer2_only_output
0,0,The recent breakthrough in quantum computing suggests that,technology,0.2509,"The recent breakthrough in quantum computing suggests that the first super-powerful computers will exist in the next two decades, leading to a race between the US and China.\n\nBut, while it’s widely known that the US leads the world when it comes to military spending, few know that we are also leading in the science of science itself.\n\nQuantum computing can solve certain tasks faster than current computers, and the potential of this technology is staggering. So how are the US and China going to develop it? And will quantum computers mean the end of the digital age?\n\nRead more: China could overtake US to become world’s superpower by 2049\n\nThe technology\n\nA quantum computer is essentially a computer with an unusually strong connection to the laws of nature.\n\nA classical computer is made up of bits, which are usually represented by 0 or 1. Quantum bits are 0 and 1 at the same time, meaning they can represent any number.\n\nThis allows a quantum computer to solve certain","The recent breakthrough in quantum computing suggests that the technology may be able to be the key to cracking some of the most computationally complex encryption in the history of the world. But what does this mean? Are computers able to hack into the internet? And what would be the consequences?\n\nTo address the second question, let’s look first at what the internet really is. The internet allows you to be connected to computers all over the world. This gives the internet the ability to share information between computers. The internet has no central location, no internet address; it uses a network of computers. This means the internet contains different locations.\n\nTo be connected to the internet you either have to be a user or have some way to be connected. That would be either a device which contains the internet address or some other method of connecting. That way you are able to be contacted on the internet. The internet is able to share information between computers because the computers on the internet are connected to the internet. This is done in a",The recent breakthrough in quantum computing suggests that the algorithms for certain types of algorithms may become computable. The algorithms that are computable are the ones that involve an arbitrarily large number of quantum bits. Such algorithms would be computable using silicon silicon technology. A variety of techniques have been developed to increase the number of quantum bits in a computer.\nTypically the algorithms which are computable may be divided into two broad categories. The first category of algorithms which would be computable using silicon silicon technology would be the algorithms which are based on the classical version of the algorithm. Examples would be the algorithms which would be used to translate between the quantum states of two photon and two electron systems. A second broad category of algorithms which would be computable would be the algorithms which would be based on the quantum version of the algorithm. For example the algorithms which would be based on the algorithms which would be based on the algorithms which would be based on the algorithms which would be based on the algorithms which would be based on the algorithms which would be based on the,"The recent breakthrough in quantum computing suggests that the first practical quantum computer could be built using a device made of silicon dioxide (SOI), the same material that is used in most modern microchips and ICs. However, the device needs to be protected from its surroundings so that it does not interact with the external environment. The silicon oxide (SiO) film is a good material to use for this purpose.\n\nOne way to do this is to make SOI quantum transistors by first making a SOI wafer (a silicon substrate), then grow

## Step 4 - Batch run: one LONGER prompt per topic

Long prompts route unambiguously and long generations give both detectors enough
scored tokens (the length lesson from notebook 08). 8 prompts x 4 generations x
200 tokens - budget ~25-35 min on a Colab T4.

In [13]:
LONG_PROMPTS_PER_TOPIC = {
    "technology": ("The new AI chip announced this week runs twice as fast as the previous "
                   "generation while drawing far less power, and analysts say its improved memory "
                   "bandwidth could reshape the entire market for data center hardware."),
    "medicine": ("Doctors tested the new vaccine across three hospitals last winter, enrolling "
                 "thousands of volunteers in a randomized trial that carefully tracked side effects "
                 "and antibody levels for six months after the final dose."),
    "sports": ("The cricket team chased down 280 runs in the final match and won the trophy on the "
               "very last ball, capping a tournament full of close finishes and record-breaking "
               "batting performances from younger players."),
    "politics": ("The government passed a controversial new law just before the election, and the "
                 "opposition immediately challenged it in court, arguing that the rushed midnight "
                 "vote ignored decades of constitutional precedent."),
    "science": ("Scientists discovered an unexpected new particle at the collider last spring, and "
                "independent teams have spent months reanalyzing the collision data to confirm "
                "whether the signal survives every statistical check."),
    "entertainment": ("The movie sequel broke box office records opening weekend, selling out "
                      "late-night screenings in dozens of cities and earning praise for its "
                      "practical effects despite mixed reviews from longtime fans."),
    "finance": ("Central bank rates pushed markets down again this quarter, and investors rotated "
                "out of growth stocks into bonds as inflation reports kept coming in hotter than "
                "most economists had forecast earlier in the year."),
    "history": ("Ancient Rome fell after centuries of slow decline, as repeated invasions, economic "
                "collapse and constant political instability gradually dismantled an empire that "
                "had once governed the entire Mediterranean world."),
}

results_df = dwm.watermark(list(LONG_PROMPTS_PER_TOPIC.values()),
                           include_single_layers=True)
results_df.head()

,id,prompt,topic,topic_score,plain_output,dual_watermarked_output,layer1_only_output,layer2_only_output
0,0,"The new AI chip announced this week runs twice as fast as the previous generation while drawing far less power, and analysts say its improved memory bandwidth could reshape the entire market for data center hardware.",technology,0.2887,"The new AI chip announced this week runs twice as fast as the previous generation while drawing far less power, and analysts say its improved memory bandwidth could reshape the entire market for data center hardware.\n\nCalled the IBM BlueGene/Q, the new chip's new architecture features 16 billion transistors and, for the first time, comes with a built-in memory controller. The result is a chip that is four times more powerful than the previous generation, while consuming half the power.\n\nThat power savings will come in handy for Big Blue, which has struggled with a lackluster PC business in recent years. The company reported this week that it has sold fewer than half as many PCs this quarter as it did a year ago, despite having the world's largest manufacturing plant, one that produces over 30,000 PCs a day.\n\n""BlueGene/Q really represents a fundamental improvement to IBM's competitive position in the server market,"" said Mike Bell, director of research for Gartner.\n\nThe chip's new memory controller is also ""a first for IBM,"" Bell said, as it is the first time a server has been able to provide enough memory bandwidth to perform","The new AI chip announced this week runs twice as fast as the previous generation while drawing far less power, and analysts say its improved memory bandwidth could reshape the entire market for data center hardware.\n\nIntel introduced the Xeon Scalable Processor on Monday, which uses two different versions of the same 28nm fabrication technology that’s used on the processors the company has been manufacturing for the past four years. The Xeon Scalable Processor is the latest version of the Xeon processor designed to run faster, be better integrated with a host computer and, in the case of the Xeon processor the company introduced Monday, be faster than the 28nm architecture.\n\n“Xeon processors have been really on the cutting edge,” says David Bishop, vice president of technology architecture at research and advisory firm Directions on Microsoft. “But what Intel has done is fundamentally change what they’re doing.”\n\nIntel uses the architecture called “Cascade Lake” on the Xeon processors. That architecture uses two different manufacturing technologies. The older architecture, named “Lake Shore,” uses the 28nm fabrication technology, while the Cascade Lake architecture uses the older 22nm","The new AI chip announced this week runs twice as fast as the previous generation while drawing far less power, and analysts say its improved memory bandwidth could reshape the entire market for data center hardware.\n\nIntel has spent the last two years developing what it describes as the most advanced deep learning chip on the market, which the company previewed on Monday. The chip, which was named the ""Ponte Vecchio,"" runs on two independent processors that work together on deep learning workloads. The two processors work together in a sort of ""cloud on a chip,"" using Intel's Xeon processors. That means the two processors share memory and processing. The Ponte Vecchio breaks new ground on the architecture of deep learning chips.\n\nThe two processors also work closely together, and Intel said that the two processors share the same network. That network effectively enables the two processors to work on deep learning workloads.\n\n""These two processors work together on the same network,"" Intel CEO Brian Krzanich said during the company's second quarter earnings call on Tuesday. ""This network effectively enables the two processors to work on the same deep learning workloads.\n\n""This is unique architecture,""","The new AI chip announced this week runs twi

In [14]:
import gc

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"VRAM allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB | "
          f"reserved: {torch.cuda.memory_reserved()/1e9:.2f} GB")
else:
    print("no CUDA device - skipping cache clear")

VRAM allocated: 6.34 GB | reserved: 7.06 GB


## Step 5 - Cross-detection with BOTH detectors

Every output column is scored twice:

- `topic_detection.detect_topic_watermark` — re-infers the topic, counts hits on
  the static list (z + binomial p)
- `kgw_detection.detect_private_watermark` — replays `derive_set` per position
  with the owner's key (ownership score + binomial p)

Expected pattern:

| kind | topic alarm | kgw alarm |
|---|---|---|
| plain | off | off |
| layer1_only | **on** | off |
| layer2_only | off | **on** |
| dual | **on** | **on** |

In [15]:
from src.detection import kgw_detection, topic_detection

tstate = topic_detection.prepare(model, tokenizer,
                                 greenlist_dir=GREENLIST_DIR, split=SPLIT)

KINDS = [("plain_output", "plain"),
         ("layer1_only_output", "layer1 (topic)"),
         ("layer2_only_output", "layer2 (kgw)"),
         ("dual_watermarked_output", "dual")]

rows = []
for _, r in results_df.iterrows():
    for col, label in KINDS:
        text = r[col]

        t_res = topic_detection.detect_topic_watermark(
            text, tokenizer, tstate, vocab_size=VOCAB_SIZE)
        k_res = kgw_detection.detect_private_watermark(
            text=text,
            tokenizer=tokenizer,
            key=key,
            vocab_size=VOCAB_SIZE,
            green_fraction=config["GREEN_FRACTION"],
            prev_token_size=config["PREV_TOKEN_SIZE"],
            threshold=config["DETECTION_THRESHOLD"],
            p_value_threshold=config["P_VALUE_THRESHOLD"],
        )

        rows.append({
            "id": r["id"],
            "gen_topic": r["topic"],
            "kind": label,
            "det_topic": t_res["topic"],
            "topic_z": t_res["z_score"],
            "topic_p": t_res["p_value"],
            "topic_ok": t_res["confirmed"],
            "kgw_score": k_res["ownership_score"],
            "kgw_p": k_res["p_value"],
            "kgw_ok": k_res["confirmed"],
        })

det_df = pd.DataFrame(rows)
pd.set_option("display.width", 160)
det_df

,id,gen_topic,kind,det_topic,topic_z,topic_p,topic_ok,kgw_score,kgw_p,kgw_ok
0,0,technology,plain,history,0.4874,3.415973e-01,False,0.482906,0.721804,False
1,0,technology,layer1 (topic),technology,8.3142,1.462476e-14,True,0.538462,0.133195,False
2,0,technology,layer2 (kgw),history,-0.1199,5.777122e-01,False,0.568376,0.021240,False
3,0,technology,dual,technology,7.5380,2.202293e-12,True,0.581197,0.007697,False
4,1,history,plain,history,-1.4760,9.478420e-01,False,0.523810,0.255328,False
5,1,history,layer1 (topic),history,7.4868,4.758584e-11,True,0.497835,0.552327,False
6,1,history,layer2 (kgw),history,-0.6612,7.740038e-01,False,0.571429,0.017513,False
7,1,history,dual,history,6.8757,1.051284e-09,True,0.580087,0.008832,False
8,2,history,plain,history,-2.9537,9.995990e-01,False,0.478632,0.763919,False
9,2,history,layer1 (topic),history,6.5599,4.723614e-09,True,0.487179,0.676335,False


In [16]:
# aggregate view: mean statistics and how many texts trip each alarm per kind
summary = det_df.groupby("kind").agg(
    mean_topic_z=("topic_z", "mean"),
    mean_kgw_score=("kgw_score", "mean"),
    topic_alarm_rate=("topic_ok", "mean"),
    kgw_alarm_rate=("kgw_ok", "mean"),
).round(3)
summary

,mean_topic_z,mean_kgw_score,topic_alarm_rate,kgw_alarm_rate
kind,,,,
dual,7.947,0.600,1.0,0.500
layer1 (topic),9.406,0.500,1.0,0.000
layer2 (kgw),0.031,0.564,0.0,0.125
plain,-0.615,0.502,0.0,0.000


## Step 6 - Persist results

In [17]:
import os

os.makedirs("data/example", exist_ok=True)
results_df.to_csv("data/example/dual_layer_example.csv", index=False)
det_df.to_csv("data/example/dual_layer_detection.csv", index=False)
print("saved data/example/dual_layer_example.csv")
print("saved data/example/dual_layer_detection.csv")

saved data/example/dual_layer_example.csv
saved data/example/dual_layer_detection.csv


## Export — `src/watermark/dual_layer.py`

Freezes this notebook into the reusable module, mirroring `first_layer.py`:

- `DualLayerWatermarkProcessor(...)` — both boosts in one processor, `w1`/`w2` ablation switches
- `DualWaterMarking(model, tokenizer[, key])` — key optional (reuses `WATERMARK_SECRET_KEY`
  from the environment or mints + stores a new one)
- `.watermark_text(prompt)` — single call: just the dual-watermarked text for a prompt
- `.generate(prompt)` — plain vs dual-watermarked pair (+ topic routing)
- `.extract_topic(prompt)` · `.watermark(prompts, include_single_layers=...)` — batch with single-layer ablations

Path assumes the notebook runs from the repo root (after `%cd` on Colab).


In [ ]:
%%writefile src/watermark/dual_layer.py
'''Dual-layer watermarking: public topic boost + private KGW boost in one pass.

Layer 1 (public):  the prompt's topic is inferred by cosine similarity in OPT's
own embedding space; that topic's static uniform greenlist
(data/greenlist/<topic>.csv) is boosted by delta_public at every step.
Layer 2 (private): a keyed rolling greenlist derived per position via
key_manager.derive_set(key, prev_tokens) and boosted by delta_private.

One LogitsProcessor applies both; w1/w2 switch layers off for ablations
(layer1-only = w2=0, layer2-only = w1=0, dual = w1=w2).

Everything runs in the tokenizer id-space (len(tokenizer)), shared with the
greenlist CSVs and both detectors.
'''
import os

import pandas as pd
import torch
from transformers import LogitsProcessor

from src.utils.key_manager import derive_set, generate_key
from src.watermark.first_layer import build_topic_matrix, load_uniform_greenlists

DEFAULT_GREENLIST_DIR = "data/greenlist"


class DualLayerWatermarkProcessor(LogitsProcessor):
    """
    Applies both watermark boosts to the next-token logits at every step.
    boosted_logit = base_logit + w1 * delta_public + w2 * delta_private
    """

    def __init__(self, topic_green_ids, key, vocab_size, green_fraction=0.5,
                 delta_public=2.0, delta_private=0.7, w1=1.0, w2=1.0,
                 prev_token_size=5):
        self.topic_ids = torch.tensor(topic_green_ids, dtype=torch.long)
        self.key = key
        self.vocab_size = vocab_size
        self.green_fraction = green_fraction
        self.delta_public = delta_public
        self.delta_private = delta_private
        self.w1 = w1
        self.w2 = w2
        self.prev_token_size = prev_token_size

    def __call__(self, input_ids, scores):
        if self.w1 != 0.0:                       # layer 1: fixed topic greenlist
            scores[..., self.topic_ids.to(scores.device)] += self.w1 * self.delta_public

        if self.w2 != 0.0:                       # layer 2: keyed rolling greenlist
            for b in range(input_ids.shape[0]):
                preferred = derive_set(
                    vocab_size=self.vocab_size,
                    green_fraction=self.green_fraction,
                    key=self.key,
                    prev_tokens_size=self.prev_token_size,
                    prev_tokens=input_ids[b].tolist(),
                )
                idx = torch.tensor(list(preferred), dtype=torch.long,
                                   device=scores.device)
                scores[b, idx] += self.w2 * self.delta_private
        return scores


class DualWaterMarking:
    """End-to-end dual watermarking over the uniform static greenlists.

    watermark_text(prompt)          -> str   (dual-watermarked continuation only)
    generate(prompt)                -> dict  {"topic", "topic_score",
                                              "plain_output",
                                              "dual_watermarked_output"}
    extract_topic(prompt)           -> (topic, ranked [(topic, cos), ...])
    watermark(prompts, ...)         -> pd.DataFrame batch (+ single-layer
                                       ablations when include_single_layers)
    """

    def __init__(self, model, tokenizer, key=None,
                 greenlist_dir=DEFAULT_GREENLIST_DIR, split="all",
                 greenlists=None, normed_embeddings=None, topic_matrix=None,
                 green_fraction=0.5, prev_token_size=5,
                 delta_public=2.0, delta_private=0.7, w1=1.0, w2=1.0,
                 max_new_tokens=200, temperature=1.0, top_p=0.9, seed=None):
        self.model = model.eval()
        self.tokenizer = tokenizer
        self.split = split
        self.vocab_size = len(tokenizer)      # id-space shared by lists + detector
        self.green_fraction = green_fraction
        self.prev_token_size = prev_token_size
        self.delta_public = delta_public
        self.delta_private = delta_private
        self.w1, self.w2 = w1, w2
        self.max_new_tokens = max_new_tokens
        self.temperature = temperature
        self.top_p = top_p

        if seed is not None:
            torch.manual_seed(seed)

        if key is None:                       
            key = os.getenv("WATERMARK_SECRET_KEY") or generate_key()
        self.key = key

        if greenlists is None:
            greenlists = load_uniform_greenlists(greenlist_dir)
        self.greenlists = greenlists
        self.topics = sorted(greenlists)

        if normed_embeddings is None:
            emb = self.model.get_input_embeddings().weight.detach().float()
            normed_embeddings = emb / emb.norm(dim=1, keepdim=True)
        self.normed_embeddings = normed_embeddings
        self.topic_matrix = (topic_matrix if topic_matrix is not None
                             else build_topic_matrix(self.model, self.tokenizer, self.topics))

    @torch.no_grad()
    def extract_topic(self, prompt):
        """Mean-pool prompt embeddings, cosine vs each topic vector."""
        ids = self.tokenizer.encode(prompt, add_special_tokens=False)
        vec = self.normed_embeddings[ids].mean(dim=0)
        vec = vec / vec.norm()
        scores = (self.topic_matrix @ vec).tolist()
        ranked = sorted(zip(self.topics, scores), key=lambda x: -x[1])
        return ranked[0][0], ranked

    def _make_processor(self, topic, w1, w2):
        """Both layers in one processor; set a weight to 0 to drop that layer."""
        return DualLayerWatermarkProcessor(
            topic_green_ids=self.greenlists[topic][self.split],
            key=self.key,
            vocab_size=self.vocab_size,
            green_fraction=self.green_fraction,
            delta_public=self.delta_public,
            delta_private=self.delta_private,
            w1=w1, w2=w2,
            prev_token_size=self.prev_token_size,
        )

    def _generate(self, inputs, processor=None):
        kwargs = dict(max_new_tokens=self.max_new_tokens, do_sample=True,
                      temperature=self.temperature, top_p=self.top_p)
        if processor is not None:
            kwargs["logits_processor"] = [processor]
        with torch.no_grad():
            out = self.model.generate(**inputs, **kwargs)
        return self.tokenizer.decode(out[0], skip_special_tokens=True)

    def _inputs(self, prompt):
        return self.tokenizer(prompt, return_tensors="pt").to(self.model.device)

    def watermark_text(self, prompt, topic=None):
        """Single-shot dual watermark: returns ONLY the watermarked text.

        topic=None routes via extract_topic; pass an explicit topic to force it.
        """
        if topic is None:
            topic, _ = self.extract_topic(prompt)
        processor = self._make_processor(topic, self.w1, self.w2)
        return self._generate(self._inputs(prompt), processor)

    def generate(self, prompt, topic=None):
        """Plain vs dual-watermarked continuation of `prompt`.

        Returns {"topic", "topic_score", "plain_output",
                 "dual_watermarked_output"}.
        """
        if topic is None:
            topic, ranked = self.extract_topic(prompt)
            topic_score = round(ranked[0][1], 4)
        else:
            topic_score = None

        inputs = self._inputs(prompt)
        plain = self._generate(inputs)
        dual = self._generate(
            inputs, self._make_processor(topic, self.w1, self.w2))
        return {
            "topic": topic,
            "topic_score": topic_score,
            "plain_output": plain,
            "dual_watermarked_output": dual,
        }

    def watermark(self, prompts, include_single_layers=True):
        """Batch: one row per prompt with all four variants.

        include_single_layers adds layer1_only_output (w2=0) and
        layer2_only_output (w1=0) for ablation/evaluation.
        """
        rows = []
        for i, prompt in enumerate(prompts):
            topic, ranked = self.extract_topic(prompt)
            inputs = self._inputs(prompt)

            row = {
                "id": i,
                "prompt": prompt,
                "topic": topic,
                "topic_score": round(ranked[0][1], 4),
                "plain_output": self._generate(inputs),
                "dual_watermarked_output": self._generate(
                    inputs, self._make_processor(topic, self.w1, self.w2)),
            }
            if include_single_layers:
                row["layer1_only_output"] = self._generate(
                    inputs, self._make_processor(topic, 1.0, 0.0))
                row["layer2_only_output"] = self._generate(
                    inputs, self._make_processor(topic, 0.0, 1.0))
            rows.append(row)
        return pd.DataFrame(rows)


Overwriting src/watermark/dual_layer.py


In [ ]:
import importlib

from src.watermark import dual_layer
importlib.reload(dual_layer)

dwm_mod = dual_layer.DualWaterMarking(
    model=model,
    tokenizer=tokenizer,
    key=key,
    greenlists=greenlists,
    split=SPLIT,
    normed_embeddings=normed_embeddings,
    topic_matrix=topic_matrix,
    green_fraction=config["GREEN_FRACTION"],
    prev_token_size=config["PREV_TOKEN_SIZE"],
    delta_public=DELTA_TOPIC,
    delta_private=DELTA_PRIVATE,
    w1=W1, w2=W2,
    max_new_tokens=MAX_NEW_TOKENS,
)

PROMPT = LONG_PROMPTS_PER_TOPIC["politics"]

torch.manual_seed(SEED)
solo_a = dwm_mod.watermark_text(PROMPT)
torch.manual_seed(SEED)
solo_b = dwm_mod.watermark_text(PROMPT)
assert solo_a == solo_b, "sampling is not deterministic under the same seed"
print("watermark_text ok |", len(tokenizer.encode(solo_a)) - len(tokenizer.encode(PROMPT)),
      "generated tokens")


pair = dwm_mod.generate(PROMPT)
assert set(pair) >= {"topic", "topic_score", "plain_output", "dual_watermarked_output"}
assert pair["plain_output"] != pair["dual_watermarked_output"]
print(f"generate ok | topic={pair['topic']} (score={pair['topic_score']})")

# 3) the solo text must trip BOTH detectors
t_det = topic_detection.detect_topic_watermark(solo_a, tokenizer, tstate, vocab_size=VOCAB_SIZE)
k_det = kgw_detection.detect_private_watermark(
    text=solo_a, tokenizer=tokenizer, key=dwm_mod.key, vocab_size=VOCAB_SIZE,
    green_fraction=config["GREEN_FRACTION"], prev_token_size=config["PREV_TOKEN_SIZE"],
    threshold=config["DETECTION_THRESHOLD"], p_value_threshold=config["P_VALUE_THRESHOLD"])
print(f"topic det: z={t_det['z_score']:+.2f} p={t_det['p_value']:.2e} confirmed={t_det['confirmed']}")
print(f"kgw det  : score={k_det['ownership_score']:.3f} p={k_det['p_value']:.2e} confirmed={k_det['confirmed']}")
print("\nDUAL TEXT:\n" + solo_a[:400] + "...")